# Verification Notebook: Urban Heat Island Analytics (Issue #33, Spec 60)

สมุดบันทึกนี้ใช้ตรวจสอบความถูกต้องของโมดูล `cloud/analytics/stats/` เทียบกับการคำนวณมือ และสร้างกราฟวินิจฉัยสำหรับประกอบรายงาน:
1. **ความซ้ำได้และวิธีทางสถิติมาตรฐาน (S1–S8):** OLS, VIF, Bootstrap 95% CI, Spatial Block CV (Roberts et al., 2017) และ Random Forest Permutation Importance (`seed=42`)
2. **การวินิจฉัย Residual Moran's I (S7):** ตรวจสอบการกระจุกตัวเชิงพื้นที่ของค่าคลาดเคลื่อนเพื่อประเมินความแคบของช่วงเชื่อมั่น
3. **การเทียบอันดับ IoT กับ LST (Rank Validation):** ตรวจสอบความสอดคล้องของอันดับสูง–ต่ำ พร้อมระบุข้อจำกัดของข้อมูลภาคสนาม (1 มี.ค. 2026)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[1] if (Path.cwd().name == 'notebooks') else Path('cloud').resolve()
sys.path.insert(0, str(ROOT / 'analytics'))
sys.path.insert(0, str(ROOT / 'pipeline'))

from stats import core, runner
print('Loaded stats engine successfully from:', ROOT / 'analytics')

In [ ]:
# 1. Cross-check OLS & VIF against manual matrix algebra on a deterministic test dataset
rng = np.random.default_rng(42)
n = 200
rows = []
for i in range(n):
    c, r = i % 20, i // 20
    ndvi = float(rng.uniform(0.05, 0.65))
    ndbi = float(rng.uniform(-0.25, 0.35))
    builtup = float(np.clip(35.0 + 80.0 * ndbi + rng.normal(0, 8.0), 0, 100))
    lst = 34.5 + 6.8 * ndbi - 2.9 * ndvi + 0.015 * builtup + float(rng.normal(0, 0.4))
    rows.append({
        'cell_id': f'c{c:04d}_r{r:04d}',
        'period': 'season_2025_03-05',
        'match_type': 'window',
        'in_cover': True,
        'lst_mean': lst,
        'ndvi_mean': ndvi,
        'ndbi_mean': ndbi,
        'builtup_pct': builtup,
        'air_temp_mean': 29.5 + 0.3 * (lst - 35.0) if i < 28 else np.nan,
        'db_spl_mean': 62.0 + 0.1 * builtup if i < 28 else np.nan,
    })
df_test = pd.DataFrame(rows)

res = core.analyze_period_match(df_test, period='season_2025_03-05', match='window', seed=42, n_boot=300)
s3 = res['regression']['models'][0]
print('S3 R2:', s3['r2'], '| Spatial Block CV R2:', s3['spatial_block_cv']['cv_r2'])
pd.DataFrame(s3['terms'])[['term', 'coef', 'std_coef', 'p_value', 'vif', 'boot_ci_95_low', 'boot_ci_95_high', 'evidence_label_th']]

In [ ]:
# 2. Inspect Spatial Autocorrelation (S7) including s3_residual & IoT Rank Validation Caveats
print('--- S7 Spatial Autocorrelation ---')
display(pd.DataFrame(res['spatial_autocorr']['rows']))
for w in res['spatial_autocorr']['warnings']:
    print('WARNING:', w)

print('\n--- IoT vs LST Rank Validation ---')
val = res['correlation']['iot_lst_validation']
print(f"Matched cells: {val['n_matched_cells']}, Spearman r: {val['spearman_r']} (p={val['spearman_p']})")
for c in val['caveats']:
    print('-', c)